# 07 · Printable test sheets for the live scanner

No real green beans? Print these sheets, lay them flat in good light, and point the
**Live scan** camera at them. Each sheet shows 12 beans from the **test set** (never used
in training), spaced apart on white, so we know the true label of every bean.

1. Run cells 1–2 to make `reports/test_sheets/sheet_1.pdf`, `sheet_2.pdf` and the answer key.
2. Print at **100% / actual size** on A4 (no "fit to page" borders needed).
3. Scan each sheet with the app. Tap **Freeze**, then **Details**, and note what the app
   says for each position (row 1 = top, column 1 = left, as you look at the sheet).
4. Fill `reports/test_sheets/live_results.csv` and run section 3 for the live accuracy.

*Printed beans are bigger than real ones. That's fine: the app's distance guidance works
from how big beans look on screen.*

In [ ]:
# --- SETUP: run this cell first -----------------------------------------
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # Change this if you uploaded the project folder somewhere else in your Drive:
    PROJECT = Path("/content/drive/MyDrive/kapeklasipika")
    %pip install -q imagehash
else:
    # Running on your laptop: walk up from this notebook until we find config/config.yaml
    PROJECT = Path.cwd()
    while not (PROJECT / "config" / "config.yaml").exists():
        PROJECT = PROJECT.parent

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

from src.config import load_config, use_local_copy
cfg = load_config()
print("Project folder:", PROJECT)

In [ ]:
import numpy as np
import pandas as pd
from PIL import Image
from src.data_loading import clean_split_files, CLASS_NAMES

OUT = PROJECT / "reports" / "test_sheets"; OUT.mkdir(parents=True, exist_ok=True)
paths, labels = clean_split_files(cfg, verbose=False)["test"]
test = pd.DataFrame({"file": paths, "label": [CLASS_NAMES[i] for i in labels]})

## 1. Pick 24 test beans (6 per class), shuffled, fixed seed so it's reproducible

In [ ]:
picked = (test.groupby("label").sample(6, random_state=7)
              .sample(frac=1, random_state=7).reset_index(drop=True))
picked["sheet"] = picked.index // 12 + 1
picked["row"] = (picked.index % 12) // 3 + 1
picked["col"] = picked.index % 3 + 1
picked.head(12)

## 2. Build the A4 sheets (300 dpi) and the answer key

In [ ]:
DPI = 300
A4 = (int(8.27 * DPI), int(11.69 * DPI))          # 2481 x 3507 px
TILE = 620                                         # ~5.2 cm per bean photo
GAP_X = (A4[0] - 3 * TILE) // 4
GAP_Y = (A4[1] - 4 * TILE) // 5

sheets = []
for s in (1, 2):
    page = Image.new("RGB", A4, "white")
    for _, r in picked[picked["sheet"] == s].iterrows():
        tile = Image.open(r.file).convert("RGB").resize((TILE, TILE), Image.BICUBIC)
        x = GAP_X + (r.col - 1) * (TILE + GAP_X)
        y = GAP_Y + (r.row - 1) * (TILE + GAP_Y)
        page.paste(tile, (x, y))
    page.save(OUT / f"sheet_{s}.pdf", resolution=DPI)
    page.save(OUT / f"sheet_{s}.png")
    sheets.append(page)

key = picked[["sheet", "row", "col", "label", "file"]]
key.to_csv(OUT / "answer_key.csv", index=False)
template = key[["sheet", "row", "col"]].assign(app_says="", app_count_on_screen="")
if not (OUT / "live_results.csv").exists():
    template.to_csv(OUT / "live_results.csv", index=False)
print("Saved to", OUT)
display(sheets[0].resize((A4[0] // 6, A4[1] // 6)))

## 3. Live accuracy (after you've scanned the sheets)
In `live_results.csv`, write what the app showed for each bean in `app_says`
(defect / longberry / peaberry / premium / not a bean / touching / missed), and the total
bean count the app showed for that sheet in `app_count_on_screen` (same number on each row
of a sheet is fine).

In [ ]:
res = pd.read_csv(OUT / "live_results.csv").merge(key, on=["sheet", "row", "col"])
res["app_says"] = res["app_says"].astype(str).str.strip().str.lower()
done = res[res["app_says"].ne("") & res["app_says"].ne("nan")]
if len(done):
    found = done[~done["app_says"].isin(["missed"])]
    acc = (found["app_says"] == found["label"]).mean()
    print(f"Beans detected: {len(found)}/{len(done)}")
    print(f"Live classification accuracy on detected beans: {acc:.1%}")
    display(pd.crosstab(done["label"], done["app_says"]))
    counts = done.groupby("sheet")["app_count_on_screen"].first()
    print("Count shown by the app per sheet (true count is 12):", counts.to_dict())
else:
    print("Fill live_results.csv first.")